# Merge the predicted results back to the original CRS 

In [ ]:
import pandas as pd

# Load the CRS data from parquet file
crs = pd.read_parquet("../../data/raw/CRS.parquet")

In [ ]:
# Read minig_sets from feather file
stat_mining_set = pd.read_feather("../../data/processed/title_matched/stat_to_mine_before_update.feather")
gen_mining_set = pd.read_feather("../../data/processed/title_matched/gen_to_mine.feather")

# Load the unlabled_predicted data from feather file
unlabeled_predicted_stat = pd.read_feather("../../data/processed/predicted/unlabeled_predicted_large_v4.feather")
unlabeled_predicted_gen = pd.read_feather("../../data/processed/predicted/unlabeled_predicted_gen.feather")

# Load conflicting_descr_stat_predicted from feather file
conflicting_stat_predicted = pd.read_feather("../../data/processed/predicted/conflicting_stat_predicted.feather")
conflicting_gen_predicted = pd.read_feather("../../data/processed/predicted/conflicting_gen_predicted.feather")

Add keywords back to the raw CRS data

In [ ]:
# Load the matched CRS data from feather file
crs_title_matched = pd.read_feather("../../data/processed/title_matched/crs_titles_matched_wo_stopwords.feather")

# Keep only the columns project_title, stat_keywords, stat_blacklist, gen_keywords, stat_acronyms, gen_acronyms
crs_title_matched = crs_title_matched[['project_title', 'language', 'stat_keywords', 'stat_blacklist', 'gen_keywords', 'stat_acronyms', 'gen_acronyms']]

# Rename lanaguage to title_language
crs_title_matched.rename(columns={'language': 'title_language'}, inplace=True)

# Left join matched CRS data to the the raw CRS data based on the project_title column
crs = pd.merge(crs, crs_title_matched, on='project_title', how='left', validate='m:1')

del crs_title_matched

In [ ]:
# Merge the stat_mining_set with the unlabeled_predicted data on 'text_mining_description'
stat_mining_predicted = pd.merge(
    stat_mining_set,
    unlabeled_predicted_stat[['text_mining_description', 'probability_is_statistics']],
    on='text_mining_description',
    how='left'
)

gen_mining_predicted = pd.merge(
    gen_mining_set,
    unlabeled_predicted_gen[['text_mining_description', 'probability_is_gender']],
    on='text_mining_description',
    how='left',
    validate='m:1'
)

del stat_mining_set, unlabeled_predicted_stat, gen_mining_set, unlabeled_predicted_gen

# If is_statistics is True, set probability_is_statistics to 1
#stat_mining_predicted.loc[stat_mining_predicted['is_statistics'] == True, 'probability_is_statistics'] = 1
stat_mining_predicted.loc[stat_mining_predicted['is_mining'] == True, 'probability_is_statistics'] = 0